# 01 — Data Preprocessing and Reproducible Split

This notebook prepares the dataset once and saves the exact train/validation/test split.

Why this notebook exists:
- Both MobileNetV2 and ResNet18 should use exactly the same images.
- The split files are saved as CSV files.
- Later model notebooks should load these CSV files instead of creating a new split again.

Outputs:
- `results/split_train.csv`
- `results/split_val.csv`
- `results/split_test.csv`
- `results/dataset_class_counts.csv`
- `results/split_class_counts.csv`


In [ ]:
!pip -q install kagglehub

import random
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

project_root = Path(".")
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

print("Results will be saved in:", results_dir.resolve())

## 1. Download and locate dataset

In [ ]:
import kagglehub

dataset_path = Path(kagglehub.dataset_download("tawsifurrahman/covid19-radiography-database"))

possible_root = dataset_path / "COVID-19_Radiography_Dataset"
data_root = possible_root if possible_root.exists() else dataset_path

print("Dataset path:", dataset_path)
print("Data root:", data_root)

## 2. Build image path and label dataframe

In [ ]:
class_names = ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]

records = []

for class_name in class_names:
    class_dir = data_root / class_name
    image_dir = class_dir / "images"
    if not image_dir.exists():
        image_dir = class_dir

    image_paths = []
    for ext in ["*.png", "*.jpg", "*.jpeg"]:
        image_paths.extend(image_dir.glob(ext))

    for path in image_paths:
        records.append({
            "path": str(path),
            "label": class_name
        })

df = pd.DataFrame(records)

print("Total images found:", len(df))
display(df.head())

original_counts = df["label"].value_counts().loc[class_names]
display(original_counts)

original_counts.to_csv(results_dir / "dataset_original_class_counts.csv", header=["count"])

## 3. Basic image checks

In [ ]:
# Check image sizes and modes before preprocessing.
# This confirms that images are consistent and identifies grayscale/RGB storage.

sizes = []
modes = []

for path in df["path"].values:
    with Image.open(path) as img:
        sizes.append(img.size)
        modes.append(img.mode)

size_counts = pd.Series(sizes).value_counts()
mode_counts = pd.Series(modes).value_counts()

print("Image size counts:")
display(size_counts.head(10))

print("Image mode counts:")
display(mode_counts)

size_counts.to_csv(results_dir / "image_size_counts.csv", header=["count"])
mode_counts.to_csv(results_dir / "image_mode_counts.csv", header=["count"])

## 4. Exact duplicate detection with SHA-256

In [ ]:
# SHA-256 detects exact duplicate files.
# Exact duplicates are removed before splitting to reduce data leakage risk.

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

df["sha256"] = df["path"].apply(sha256_file)

duplicate_mask = df["sha256"].duplicated(keep="first")
n_duplicates = int(duplicate_mask.sum())

duplicates_df = df[duplicate_mask].copy()
df_clean = df.drop_duplicates(subset="sha256", keep="first").reset_index(drop=True)

print("Original images:", len(df))
print("Exact duplicates removed:", n_duplicates)
print("Images after duplicate removal:", len(df_clean))

clean_counts = df_clean["label"].value_counts().loc[class_names]
display(clean_counts)

duplicates_df.to_csv(results_dir / "exact_duplicate_files_removed.csv", index=False)
clean_counts.to_csv(results_dir / "dataset_clean_class_counts.csv", header=["count"])

## 5. Stratified train/validation/test split

In [ ]:
# Stratified split preserves class proportions.
# The test set must remain untouched until final evaluation.

train_df, temp_df = train_test_split(
    df_clean,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=df_clean["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

split_counts = pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "validation": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts()
}).loc[class_names]

display(split_counts)

split_counts.to_csv(results_dir / "split_class_counts.csv")

## 6. Add label IDs and save split files

In [ ]:
label_to_id = {name: i for i, name in enumerate(class_names)}
id_to_label = {i: name for name, i in label_to_id.items()}

for split_df in [train_df, val_df, test_df]:
    split_df["label_id"] = split_df["label"].map(label_to_id)

label_mapping_df = pd.DataFrame([
    {"label": label, "label_id": label_id}
    for label, label_id in label_to_id.items()
])

train_df.to_csv(results_dir / "split_train.csv", index=False)
val_df.to_csv(results_dir / "split_val.csv", index=False)
test_df.to_csv(results_dir / "split_test.csv", index=False)
label_mapping_df.to_csv(results_dir / "label_mapping.csv", index=False)

print("Saved:")
print(results_dir / "split_train.csv")
print(results_dir / "split_val.csv")
print(results_dir / "split_test.csv")
print(results_dir / "label_mapping.csv")

display(label_mapping_df)

## Report note

Use this in the report:

> The dataset was inspected for class distribution, image size, image mode, and exact duplicate files. Exact duplicate images were detected using SHA-256 hashing and removed before splitting to reduce leakage risk. The cleaned dataset was split into training, validation, and test sets using a stratified 70/15/15 split. The split files were saved and reused by all model notebooks to ensure that MobileNetV2 and ResNet18 were evaluated on exactly the same test images.
